# H1 - Forme d'occurrence de *fairness* selon le type d'acteurice

Question : la forme sous laquelle *fairness* est employe depend-elle du type d'acteurice ?

On classe chaque occurrence dans une categorie de forme, lue sur l'ego-reseau AMR :

- a : invocation nue
- b : enumeration
- c : modifieur de domaine
- d : definition
- e : predicative (le terme porte un argument propre, *fairness of X* compris)
- mixte : l'occurrence remplit au moins deux des conditions b, c, d, e

puis on croise avec le type d'acteurice et on teste l'association.

Conventions decidees en amont :

- *fair-01* et *fairness* sont fusionnes en un seul concept ; la categorie se lit sur la structure, pas sur l'etiquette.
- l'unite d'analyse est l'occurrence (une ligne par noeud mot-cle).
- les occurrences sans type d'acteurice sont exclues du test.

Avertissement : ce notebook tourne sur un echantillon de test de 100 phrases, petit et desequilibre entre acteurices. Il valide la chaine de traitement et le test statistique ; il ne produit pas un resultat d'association publiable.

In [1]:
from pathlib import Path

DATA_DIR = Path(".")
PAIRED_FILE = DATA_DIR / "fairness_amr_doc_paired.csv"

# Concepts traites comme le mot-cle (fair-01 et fairness fusionnes)
KEYWORD_RE = r"^(fairness|fair-\d+)$"

N_PERMUTATIONS = 2000
RANDOM_SEED = 0

# Mapping actor_type francais -> anglais (comme dans le xlsx)
ACTOR_TYPE_MAP_FR_EN = {
    "academique": "academia",
    "societe civile": "civil society",
    "organisation internationale": "international organisation",
    "autorite nationale": "national authority",
    "secteur prive": "private sector"
}

In [3]:
import re
import pandas as pd
import numpy as np
import penman
from scipy.stats import chi2_contingency

## Regle de classification a-e

Pour chaque occurrence on ne regarde que les aretes propres du noeud mot-cle (relations vers ses parents directs, relations vers ses enfants). Un noeud peut remplir plusieurs conditions ; dans ce cas il recoit la categorie *mixte*.

- e (predicative) : le noeud a un enfant argument (:ARG0..:ARG5).
- c (modifieur de domaine) : le noeud a un enfant :mod ou :domain.
- d (definition) : le parent direct est un predicat de definition (`DEFINITION_FRAMES`) et le terme y occupe la place :ARG0 ou :ARG1.
- b (enumeration) : le parent direct est une conjonction (and / or) reliee par :opN.
- a (invocation nue) : aucune des conditions ci-dessus.

`DEFINITION_FRAMES` et la frontiere entre categories sont des decisions d'analyste, editables ci-dessous.

In [4]:
# Predicats de definition (categorie d). Liste editable : decision d'analyste.
DEFINITION_FRAMES = {"mean-01", "refer-01", "imply-01", "define-01",
                     "denote-01", "describe-01", "constitute-01"}
DEFINIENDUM = {":ARG0", ":ARG1"}   # places du terme defini sous un predicat de definition
CAT_COL = "category"

In [5]:
keyword_re = re.compile(KEYWORD_RE)
CORE_ARGS = {f":ARG{i}" for i in range(6)}   # arguments de coeur :ARG0..:ARG5

def parse_metadata(block):
    head = "\n".join(l for l in block.splitlines() if l.startswith("#"))
    sid = re.search(r"::id (\S+)", head)
    did = re.search(r"::doc_id (\S+)", head)
    snt = re.search(r"::snt (.+)", head)
    return (sid.group(1) if sid else None,
            int(did.group(1)) if did else None,
            snt.group(1).strip() if snt else "")

def canonical_edges(g):
    # remet les aretes inverses (:role-of) dans le sens tete -> dependant
    edges = []
    for s, r, t in g.edges():
        if r.endswith("-of") and r != ":consist-of":
            edges.append((t, ":" + r[1:-3], s))
        else:
            edges.append((s, r, t))
    return edges

In [6]:
def classify_block(block):
    sid, did, snt = parse_metadata(block)
    graph_str = "\n".join(l for l in block.splitlines() if not l.startswith("#"))
    g = penman.decode(graph_str)
    concept = {v: c for v, _, c in g.instances()}
    E = canonical_edges(g)
    rows = []
    kw_vars = sorted(v for v, c in concept.items() if c and keyword_re.match(c))
    for i, v in enumerate(kw_vars):
        children = [(r, dep) for h, r, dep in E if h == v]
        parents  = [(h, r)   for h, r, dep in E if dep == v]
        has_e = any(r in CORE_ARGS for r, _ in children)
        has_c = any(r in (":mod", ":domain") for r, _ in children)
        has_d = any(concept.get(h) in DEFINITION_FRAMES and r in DEFINIENDUM
                    for h, r in parents)
        has_b = any(re.match(r":op\d+$", r) and concept.get(h) in ("and", "or")
                    for h, r in parents)
        flags = [k for k, on in (("b", has_b), ("c", has_c), ("d", has_d), ("e", has_e)) if on]
        category = "a" if not flags else (flags[0] if len(flags) == 1 else "mixte")
        rows.append(dict(sample_id=sid, doc_id=did, occ=i,
                         keyword_surface=concept[v], category=category,
                         flags="".join(flags), sentence=snt))
    return rows

In [7]:
import penman

paired = pd.read_csv(PAIRED_FILE)

# Extraire les AMR et les parser
occ_rows, no_keyword = [], 0
for _, row in paired.iterrows():
    block = row["amr_penman"].strip()
    if not block:
        continue
    
    # Parser avec penman
    try:
        g = penman.decode(block)
    except Exception as e:
        print(f"Erreur parse {row['sample_id']}: {e}")
        continue
    
    concept = {v: c for v, _, c in g.instances()}
    E = canonical_edges(g)
    
    for i, v in enumerate([v for v, c in concept.items() if c and keyword_re.match(c)]):
        children = [(r, dep) for h, r, dep in E if h == v]
        parents  = [(h, r)   for h, r, dep in E if dep == v]
        has_e = any(r in CORE_ARGS for r, _ in children)
        has_c = any(r in (":mod", ":domain") for r, _ in children)
        has_d = any(concept.get(h) in DEFINITION_FRAMES and r in DEFINIENDUM
                    for h, r in parents)
        has_b = any(re.match(r":op\d+$", r) and concept.get(h) in ("and", "or")
                    for h, r in parents)
        flags = [k for k, on in (("b", has_b), ("c", has_c), ("d", has_d), ("e", has_e)) if on]
        category = "a" if not flags else (flags[0] if len(flags) == 1 else "mixte")
        occ_rows.append(dict(sample_id=row["sample_id"], doc_id=row["doc_id"], occ=i,
                             keyword_surface=concept[v], category=category,
                             flags="".join(flags), sentence=row["sentence"]))

df = pd.DataFrame(occ_rows)
df["keyword_norm"] = "fairness"   # fusion fair-01 / fairness

print(f"phrases (blocs) : {len(paired)}")
print(f"phrases sans noeud mot-cle : {paired['amr_penman'].isna().sum()}")
print(f"occurrences trouvees : {len(df)}")
print("\netiquette de surface avant fusion :")
print(df["keyword_surface"].value_counts().to_string())
print("\ndistribution des categories de forme :")
print(df["category"].value_counts().to_string())

ignoring epigraph data for duplicate triple: ('m2', ':ARG0', 'o')
ignoring epigraph data for duplicate triple: ('m2', ':ARG1', 'p3')


phrases (blocs) : 100
phrases sans noeud mot-cle : 4
occurrences trouvees : 110

etiquette de surface avant fusion :
keyword_surface
fairness    76
fair-01     34

distribution des categories de forme :
category
b        44
a        38
e        12
mixte     6
d         5
c         5


In [10]:
actor = pd.read_csv(ACTOR_MAP)[["doc_id", "actor_type", "needs_review"]]
df = df.merge(actor, on="doc_id", how="left")

n_nan = df["actor_type"].isna().sum()
print(f"occurrences sans acteurice (exclues du test) : {n_nan}")
print(f"documents acteurice marques needs_review : {int(actor['needs_review'].sum())} / {len(actor)} "
      "(etiquettes provisoires, non reverifiees)")

test_df = df.dropna(subset=["actor_type"]).copy()
print(f"occurrences retenues pour le test : {len(test_df)}")

KeyError: 'actor_type'

In [9]:
ct = pd.crosstab(test_df["category"], test_df["actor_type"])
ct

actor_type,academia,civil society,international organisation,national authority,private sector
category,,,,,
a,0,9,14,10,5
b,3,7,20,9,5
c,0,0,1,3,1
d,0,1,2,2,0
e,1,1,4,4,2
mixte,0,0,3,3,0


# Le paired file porte déjà actor_type (en français)
# Faire le mapping vers anglais et ajouter au df

actor_map = pd.DataFrame({
    "doc_id": paired["doc_id"].values,
    "actor_type_en": paired["actor_type"].map(ACTOR_TYPE_MAP_FR_EN).values,
    "needs_review": False  # placeholder, le paired est déjà vérifié par URL
})
actor_map = actor_map.drop_duplicates("doc_id")

df = df.merge(actor_map, on="doc_id", how="left", suffixes=("", "_en"))
df["actor_type"] = df["actor_type_en"]

n_nan = df["actor_type"].isna().sum()
print(f"occurrences sans acteurice (exclues du test) : {n_nan}")
print(f"documents acteurice marques needs_review : 0 / {len(actor_map)} "
      "(etiquettes verifiees par appariement URL)")

test_df = df.dropna(subset=["actor_type"]).copy()
print(f"occurrences retenues pour le test : {len(test_df)}")

In [11]:
chi2, p_asym, dof, expected = chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

# test de permutation (robuste pour cases creuses)
rng = np.random.default_rng(RANDOM_SEED)
labels_cat = test_df[CAT_COL].to_numpy()
labels_act = test_df["actor_type"].to_numpy()
count = 0
for _ in range(N_PERMUTATIONS):
    perm = rng.permutation(labels_act)
    c, _, _, _ = chi2_contingency(pd.crosstab(pd.Series(labels_cat), pd.Series(perm)))
    if c >= chi2:
        count += 1
p_perm = (count + 1) / (N_PERMUTATIONS + 1)

print(f"chi2 = {chi2:.2f}   dof = {dof}")
print(f"plus petite valeur attendue = {expected.min():.2f}  (si < 5, se fier a la permutation)")
print(f"p asymptotique (chi2)      = {p_asym:.3f}")
print(f"p permutation (B={N_PERMUTATIONS}) = {p_perm:.3f}")
print(f"V de Cramer                = {cramers_v:.3f}")

chi2 = 14.72   dof = 20
plus petite valeur attendue = 0.18  (si < 5, se fier a la permutation)
p asymptotique (chi2)      = 0.792
p permutation (B=2000) = 0.830
V de Cramer                = 0.183


## Apercu pour inspecter la regle

A relire pour valider ou corriger le classement sur des cas reels.

In [12]:
pd.set_option("display.max_colwidth", 90)
test_df[["sample_id", "category", "flags", "sentence"]].head(20)

,sample_id,category,flags,sentence
0,s000,b,b,"We contribute to fairness, equal opportunities and solidarity 5."
1,s001,a,,Universal access to education must be achieved through principles of solidarity and fa...
2,s002,a,,Most decision processes (whether using algorithms or not) exhibit bias in some form an...
3,s003,b,b,"121 political rights, safety, fairness – including fair competition – and accountability."
4,s003,e,e,"121 political rights, safety, fairness – including fair competition – and accountability."
5,s004,b,b,"24 by designing platforms to support data collection and annotation, ensuring that dat..."
6,s005,b,b,Further research is necessary to understand the implications of these potential intera...
7,s006,b,b,One of the first analyses in this r espect is a recently published report of the Europ...
8,s007,a,,23 The limitations of DPIAs have been noted in terms of promoting accountability and e...
9,s008,b,b,It can mean: access upon request to the public or authorised people; public posting of...


## Limites de cet echantillon

- Desequilibre fort entre acteurices (academia et private sector quasi vides) : le test n'a presque aucune puissance pour ces deux types.
- Petit effectif et table tres creuse : la p-value asymptotique du chi2 est peu fiable, d'ou la permutation.
- Etiquettes d'acteurice provisoires (majorite marquee needs_review), non reverifiees.
- Les occurrences d'un meme document ne sont pas independantes ; pour le corpus complet, prevoir une verification de robustesse (ponderation ou regroupement par document).
- La liste DEFINITION_FRAMES et la frontiere entre categories sont des decisions d'analyste a documenter.

En resume : ce notebook montre que la chaine tourne de bout en bout et produit table, test et taille d'effet ; il ne fournit pas un resultat interpretable sur ce jeu de test.